<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/18_hybrid_retrieval_rag/hybrid_retrieval_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U sentence-transformers transformers faiss-cpu scikit-learn sentencepiece --quiet

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
documents = [
    "Paris is the capital of France.",
    "France is located in Europe.",
    "Berlin is the capital of Germany.",
    "Python is a programming language."
]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)

In [ ]:
import numpy as np

embeddings = embed_model.encode(documents)

In [ ]:
query = "What is the capital of France?"

query_embedding = embed_model.encode([query])
query_tfidf = vectorizer.transform([query])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

semantic_scores = cosine_similarity(query_embedding, embeddings)[0]
keyword_scores = cosine_similarity(query_tfidf, tfidf_matrix)[0]

In [10]:
alpha = 0.5  # balance parameter

hybrid_scores = alpha * semantic_scores + (1 - alpha) * keyword_scores

best_index = hybrid_scores.argmax()

context = documents[best_index]

print("HYBRID RETRIEVED DOCUMENT:")
print(context)

HYBRID RETRIEVED DOCUMENT:
Paris is the capital of France.


In [11]:
prompt = f"""
Use the following context to answer the question.

Context: {context}

Question: {query}
"""

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=30)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\nFINAL ANSWER:")
print(answer)


FINAL ANSWER:
Paris
